<a href="https://colab.research.google.com/github/gabriela0402/ai-interview-evaluator/blob/main/notebooks/03_saida_estruturada.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Experimento de saída estruturada

## Objetivo

Investigar se um modelo de linguagem consegue retornar avaliações de entrevistas
em um formato estruturado, facilitando a leitura e a análise automática das notas.

Neste experimento, a avaliação deverá ser retornada em JSON, contendo as notas dos
critérios definidos na rubrica.

> Este experimento é educacional. As avaliações não devem ser usadas sozinhas para
> tomar decisões reais de contratação.


In [1]:
import json
import pandas as pd


In [6]:
from google.colab import userdata
from openai import OpenAI

token_hf = userdata.get("HF_TOKEN")

cliente_hf = OpenAI(
    api_key=token_hf,
    base_url="https://router.huggingface.co/v1"
 )

modelo_hf = "openai/gpt-oss-20b"

print("Cliente configurado!")


Cliente configurado!


In [7]:
url_dataset = (
    "https://raw.githubusercontent.com/"
    "gabriela0402/ai-interview-evaluator/"
    "main/data/sample/entrevista_dataset_inicial.jsonl"
 )

dataset = pd.read_json(url_dataset, lines=True)

resposta_teste = dataset.iloc[0]

print(resposta_teste["id"])
print(resposta_teste["pergunta"])
print(resposta_teste["resposta"])


resp_001
Conte sobre uma situação em que você precisou resolver um problema.
Durante um trabalho em grupo, percebemos que os dados estavam inconsistentes no dia anterior à apresentação. Eu conferi as fontes, encontrei os valores incorretos e organizei uma divisão de tarefas para corrigirmos tudo. Conseguimos finalizar a apresentação no prazo.


In [8]:
prompt_avaliacao_json = f"""
Avalie a resposta de entrevista abaixo.

Retorne somente um objeto JSON válido.
Não escreva explicações antes ou depois do JSON.

Use exatamente estas chaves:

{{
    "relevancia": número de 1 a 5,
    "especificidade": número de 1 a 5,
    "comunicacao": número de 1 a 5,
    "reflexao": número de 1 a 5,
    "resultado": número de 1 a 5,
    "nota_geral": número de 1 a 5,
    "pontos_fortes": ["texto"],
    "pontos_a_melhorar": ["texto"],
    "justificativa": "texto"
}}

Avalie somente o conteúdo da resposta.
Não considere nome, idade, gênero, aparência ou origem.

Pergunta:
{resposta_teste["pergunta"]}

Competência:
{resposta_teste["competencia"]}

Resposta:
{resposta_teste["resposta"]}
"""

print("Prompt criado!")


Prompt criado!


In [9]:
resultado_json = cliente_hf.chat.completions.create(
    model=modelo_hf,
    messages=[
        {
            "role": "system",
            "content": (
                "Você é um avaliador experimental. "
                "Responda somente com JSON válido."
            )
        },
        {
            "role": "user",
            "content": prompt_avaliacao_json
        }
    ],
    temperature=0,
    max_tokens=500
)

resposta_bruta = resultado_json.choices[0].message.content

print(resposta_bruta)


{"relevancia":5,"especificidade":4,"comunicacao":5,"reflexao":2,"resultado":4,"nota_geral":4,"pontos_fortes":["Comunicação clara e objetiva","Demonstrou capacidade de organização e trabalho em equipe","Conseguiu resolver o problema dentro do prazo"],"pontos_a_melhorar":["Faltou reflexão sobre aprendizados ou lições tiradas da situação","Pouca profundidade nos detalhes do processo de resolução","Não destacou especificamente a contribuição individual"],"justificativa":"A resposta atende ao requisito de descrever uma situação de resolução de problema, apresentando ações concretas e um resultado positivo. A comunicação é eficaz e o contexto é relevante. Contudo, a falta de reflexão e de detalhes mais profundos sobre o processo e o papel individual reduzem a profundidade da resposta, justificando uma pontuação de 4 no geral."}


In [10]:
resposta_limpa = resposta_bruta.strip()

if resposta_limpa.startswith("```"):
    resposta_limpa = (
        resposta_limpa
        .replace("```json", "")
        .replace("```", "")
        .strip()
    )

avaliacao_estruturada = json.loads(resposta_limpa)

print("JSON convertido com sucesso!")

display(
    pd.DataFrame([avaliacao_estruturada])
)


JSON convertido com sucesso!


,relevancia,especificidade,comunicacao,reflexao,resultado,nota_geral,pontos_fortes,pontos_a_melhorar,justificativa
0,5,4,5,2,4,4,"[Comunicação clara e objetiva, Demonstrou capa...",[Faltou reflexão sobre aprendizados ou lições ...,A resposta atende ao requisito de descrever um...


In [11]:
criterios_notas = [
    "relevancia",
    "especificidade",
    "comunicacao",
    "reflexao",
    "resultado",
    "nota_geral"
]

for criterio in criterios_notas:
    nota = avaliacao_estruturada[criterio]

    if not isinstance(nota, (int, float)):
        raise ValueError(
            f"A nota de {criterio} não é numérica."
        )

    if not 1 <= nota <= 5:
        raise ValueError(
            f"A nota de {criterio} está fora da escala."
        )

print("Avaliação JSON validada com sucesso!")


Avaliação JSON validada com sucesso!


In [20]:
class ErroJsonInvalido(Exception):
    def __init__(self, mensagem, resposta_bruta):
        super().__init__(mensagem)
        self.resposta_bruta = resposta_bruta


def avaliar_resposta_json(linha):
    prompt = f"""
Avalie a resposta de entrevista abaixo.

Retorne somente um objeto JSON válido.
Não escreva explicações antes ou depois do JSON.

Use exatamente estas chaves:

{{
    "relevancia": número de 1 a 5,
    "especificidade": número de 1 a 5,
    "comunicacao": número de 1 a 5,
    "reflexao": número de 1 a 5,
    "resultado": número de 1 a 5,
    "nota_geral": número de 1 a 5,
    "pontos_fortes": ["texto"],
    "pontos_a_melhorar": ["texto"],
    "justificativa": "texto"
}}

Avalie somente o conteúdo da resposta.
Não considere nome, idade, gênero, aparência ou origem.

Pergunta:
{linha["pergunta"]}

Competência:
{linha["competencia"]}

Resposta:
{linha["resposta"]}
"""

    resultado = cliente_hf.chat.completions.create(
        model=modelo_hf,
        messages=[
            {
                "role": "system",
                "content": (
                    "Você é um avaliador experimental. "
                    "Responda somente com JSON válido."
                )
            },
            {
                "role": "user",
                "content": prompt
            }
        ],
        temperature=0,
        max_tokens=800
    )

    resposta_bruta = (
        resultado.choices[0].message.content or ""
    ).strip()

    if resposta_bruta.startswith("```"):
        resposta_bruta = (
            resposta_bruta
            .replace("```json", "")
            .replace("```", "")
            .strip()
        )

    try:
        avaliacao = json.loads(resposta_bruta)

    except json.JSONDecodeError as erro:
        raise ErroJsonInvalido(
            f"JSON inválido ou incompleto: {erro}",
            resposta_bruta
        )

    return avaliacao, resposta_bruta


In [21]:
import time

resultados_json = []

for indice, linha in dataset.iterrows():
    print(
        f"Avaliando resposta "
        f"{indice + 1} de {len(dataset)}..."
    )

    try:
        avaliacao, resposta_bruta = avaliar_resposta_json(linha)

        resultados_json.append({
            "id": linha["id"],
            "pergunta": linha["pergunta"],
            "competencia": linha["competencia"],
            "nivel_esperado": linha["nivel_esperado"],
            "relevancia": avaliacao["relevancia"],
            "especificidade": avaliacao["especificidade"],
            "comunicacao": avaliacao["comunicacao"],
            "reflexao": avaliacao["reflexao"],
            "resultado": avaliacao["resultado"],
            "nota_geral": avaliacao["nota_geral"],
            "pontos_fortes": avaliacao["pontos_fortes"],
            "pontos_a_melhorar": avaliacao["pontos_a_melhorar"],
            "justificativa": avaliacao["justificativa"],
            "resposta_bruta": resposta_bruta,
            "status": "sucesso"
        })

    except ErroJsonInvalido as erro:
        resultados_json.append({
            "id": linha["id"],
            "pergunta": linha["pergunta"],
            "competencia": linha["competencia"],
            "nivel_esperado": linha["nivel_esperado"],
            "relevancia": None,
            "especificidade": None,
            "comunicacao": None,
            "reflexao": None,
            "resultado": None,
            "nota_geral": None,
            "pontos_fortes": None,
            "pontos_a_melhorar": None,
            "justificativa": None,
            "resposta_bruta": erro.resposta_bruta,
            "status": f"json_invalido: {erro}"
        })

    except Exception as erro:
        resultados_json.append({
            "id": linha["id"],
            "pergunta": linha["pergunta"],
            "competencia": linha["competencia"],
            "nivel_esperado": linha["nivel_esperado"],
            "relevancia": None,
            "especificidade": None,
            "comunicacao": None,
            "reflexao": None,
            "resultado": None,
            "nota_geral": None,
            "pontos_fortes": None,
            "pontos_a_melhorar": None,
            "justificativa": None,
            "resposta_bruta": None,
            "status": f"erro_api: {erro}"
        })

    time.sleep(2)

resultados_json_df = pd.DataFrame(resultados_json)

print("Processamento concluído!")
display(resultados_json_df)


Avaliando resposta 1 de 9...
Avaliando resposta 2 de 9...
Avaliando resposta 3 de 9...
Avaliando resposta 4 de 9...
Avaliando resposta 5 de 9...
Avaliando resposta 6 de 9...
Avaliando resposta 7 de 9...
Avaliando resposta 8 de 9...
Avaliando resposta 9 de 9...
Processamento concluído!


,id,pergunta,competencia,nivel_esperado,relevancia,especificidade,comunicacao,reflexao,resultado,nota_geral,pontos_fortes,pontos_a_melhorar,justificativa,resposta_bruta,status
0,resp_001,Conte sobre uma situação em que você precisou ...,resolucao_de_problemas,5,5.0,4.0,5.0,2.0,5.0,4.0,"[Clareza na descrição do problema e solução, O...","[Falta de reflexão sobre aprendizados, Pouca p...",A resposta demonstra claramente que o candidat...,"{""relevancia"":5,""especificidade"":4,""comunicaca...",sucesso
1,resp_002,Conte sobre uma situação em que você precisou ...,resolucao_de_problemas,3,4.0,2.0,3.0,1.0,2.0,2.0,"[Respondeu à pergunta, Menciona que utilizou p...",[Falta detalhes sobre o problema e os passos t...,"A resposta atende à pergunta de forma geral, m...","{""relevancia"":4,""especificidade"":2,""comunicaca...",sucesso
2,resp_003,Conte sobre uma situação em que você precisou ...,resolucao_de_problemas,1,1.0,1.0,2.0,1.0,1.0,1.0,[Honestidade],[Descrever uma situação concreta de resolução ...,A resposta não descreve nenhuma situação em qu...,"{""relevancia"":1,""especificidade"":1,""comunicaca...",sucesso
3,resp_004,Fale sobre uma situação em que você trabalhou ...,trabalho_em_equipe,5,4.0,3.0,4.0,2.0,4.0,3.0,"[Organizou reunião para alinhar tarefas, Ajudo...","[Faltou detalhar os desafios enfrentados, Não ...",A resposta demonstra ações concretas de colabo...,"{""relevancia"":4,""especificidade"":3,""comunicaca...",sucesso
4,resp_005,Fale sobre uma situação em que você trabalhou ...,trabalho_em_equipe,3,2.0,1.0,2.0,1.0,1.0,1.0,[Respondeu à pergunta],"[Falta detalhes sobre a situação, Não há refle...",A resposta é extremamente breve e não fornece ...,"{""relevancia"":2,""especificidade"":1,""comunicaca...",sucesso
5,resp_006,Fale sobre uma situação em que você trabalhou ...,trabalho_em_equipe,1,1.0,1.0,2.0,1.0,1.0,1.0,[Clarity of expression],"[Does not address teamwork, Lacks detail, No r...",The response does not address the question abo...,"{""relevancia"":1,""especificidade"":1,""comunicaca...",sucesso
6,resp_007,Como você lida com um prazo curto?,organizacao,5,4.0,3.0,4.0,3.0,3.0,4.0,"[Verifica tudo o que precisa ser feito, Separa...","[Falta detalhamento de como prioriza, Não menc...",A resposta demonstra boa organização e comunic...,"{""relevancia"":4,""especificidade"":3,""comunicaca...",sucesso
7,resp_008,Como você lida com um prazo curto?,organizacao,3,4.0,2.0,3.0,2.0,2.0,3.0,[Prioriza as tarefas mais importantes quando o...,[Falta de detalhes sobre como organiza as tare...,A resposta demonstra entendimento da necessida...,"{""relevancia"":4,""especificidade"":2,""comunicaca...",sucesso
8,resp_009,Como você lida com um prazo curto?,organizacao,1,NaN,NaN,NaN,NaN,NaN,NaN,None,None,None,None,erro_api: Error code: 402 - {'error': 'You hav...


In [22]:
def reprocessar_json_invalido(
    resultados_df,
    dataset,
    cliente,
    modelo,
    intervalo=2
):
    """
    Reprocessa apenas as respostas que tiveram JSON inválido.

    Não reprocessa respostas bem-sucedidas nem erros de API.
    """

    resultados_atualizados = resultados_df.copy()

    ids_com_json_invalido = resultados_atualizados.loc[
        resultados_atualizados["status"].astype(str).str.startswith(
            "json_invalido"
        ),
        "id"
    ].tolist()

    print(
        f"Respostas com JSON inválido para reprocessar: "
        f"{len(ids_com_json_invalido)}"
    )

    for id_resposta in ids_com_json_invalido:
        linha = dataset[
            dataset["id"] == id_resposta
        ].iloc[0]

        print(f"Reprocessando {id_resposta}...")

        try:
            avaliacao, resposta_bruta = avaliar_resposta_json(
                linha
            )

            indice_resultado = resultados_atualizados[
                resultados_atualizados["id"] == id_resposta
            ].index[0]

            resultados_atualizados.loc[
                indice_resultado,
                "relevancia"
            ] = avaliacao["relevancia"]

            resultados_atualizados.loc[
                indice_resultado,
                "especificidade"
            ] = avaliacao["especificidade"]

            resultados_atualizados.loc[
                indice_resultado,
                "comunicacao"
            ] = avaliacao["comunicacao"]

            resultados_atualizados.loc[
                indice_resultado,
                "reflexao"
            ] = avaliacao["reflexao"]

            resultados_atualizados.loc[
                indice_resultado,
                "resultado"
            ] = avaliacao["resultado"]

            resultados_atualizados.loc[
                indice_resultado,
                "nota_geral"
            ] = avaliacao["nota_geral"]

            resultados_atualizados.loc[
                indice_resultado,
                "pontos_fortes"
            ] = str(avaliacao["pontos_fortes"])

            resultados_atualizados.loc[
                indice_resultado,
                "pontos_a_melhorar"
            ] = str(avaliacao["pontos_a_melhorar"])

            resultados_atualizados.loc[
                indice_resultado,
                "justificativa"
            ] = avaliacao["justificativa"]

            resultados_atualizados.loc[
                indice_resultado,
                "resposta_bruta"
            ] = resposta_bruta

            resultados_atualizados.loc[
                indice_resultado,
                "status"
            ] = "sucesso_reprocessado"

        except ErroJsonInvalido as erro:
            indice_resultado = resultados_atualizados[
                resultados_atualizados["id"] == id_resposta
            ].index[0]

            resultados_atualizados.loc[
                indice_resultado,
                "resposta_bruta"
            ] = erro.resposta_bruta

            resultados_atualizados.loc[
                indice_resultado,
                "status"
            ] = f"json_invalido_novamente: {erro}"

        except Exception as erro:
            indice_resultado = resultados_atualizados[
                resultados_atualizados["id"] == id_resposta
            ].index[0]

            resultados_atualizados.loc[
                indice_resultado,
                "status"
            ] = f"erro_api_no_reprocessamento: {erro}"

        time.sleep(intervalo)

    return resultados_atualizados


In [23]:
resultados_json_df = reprocessar_json_invalido(
    resultados_df=resultados_json_df,
    dataset=dataset,
    cliente=cliente_hf,
    modelo=modelo_hf,
    intervalo=2
)

display(
    resultados_json_df[
        ["id", "status"]
    ]
)


Respostas com JSON inválido para reprocessar: 0


,id,status
0,resp_001,sucesso
1,resp_002,sucesso
2,resp_003,sucesso
3,resp_004,sucesso
4,resp_005,sucesso
5,resp_006,sucesso
6,resp_007,sucesso
7,resp_008,sucesso
8,resp_009,erro_api: Error code: 402 - {'error': 'You hav...


In [26]:
resultados_json_df.to_json(
    "resultados_saida_estruturada_reprocessados.jsonl",
    orient="records",
    lines=True,
    force_ascii=False
)

resultados_json_df.to_csv(
    "resultados_saida_estruturada_reprocessados.csv",
    index=False,
    encoding="utf-8-sig"
)


In [17]:
resultados_json_df.to_json(
    "resultados_saida_estruturada_parcial.jsonl",
    orient="records",
    lines=True,
    force_ascii=False
)

resultados_json_df.to_csv(
    "resultados_saida_estruturada_parcial.csv",
    index=False,
    encoding="utf-8-sig"
)

print("Resultados parciais salvos.")


Resultados parciais salvos.


In [18]:
resultados_json_sucesso = resultados_json_df[
    resultados_json_df["status"] == "sucesso"
].copy()

display(resultados_json_sucesso)


,id,pergunta,competencia,nivel_esperado,relevancia,especificidade,comunicacao,reflexao,resultado,nota_geral,pontos_fortes,pontos_a_melhorar,justificativa,status
0,resp_001,Conte sobre uma situação em que você precisou ...,resolucao_de_problemas,5,5.0,4.0,5.0,2.0,4.0,4.0,"[Comunicação clara e objetiva, Demonstrou capa...",[Faltou reflexão sobre aprendizados ou lições ...,A resposta atende ao requisito de descrever um...,sucesso
4,resp_005,Fale sobre uma situação em que você trabalhou ...,trabalho_em_equipe,3,2.0,1.0,2.0,1.0,1.0,1.0,[Respondeu à pergunta],"[Falta detalhes sobre a situação, Não há refle...",A resposta é extremamente breve e não fornece ...,sucesso
7,resp_008,Como você lida com um prazo curto?,organizacao,3,4.0,2.0,3.0,3.0,2.0,3.0,"[Prioriza tarefas importantes, Reconhece limit...",[Falta detalhamento de estratégias específicas...,A resposta demonstra entendimento básico da qu...,sucesso
